In [6]:
from user_api import *

In [53]:
import pandas as pd

try:
    df = pd.read_parquet("../data/raw/games.parquet")
    print("Cargado desde Parquet.")
except Exception:
    df = pd.read_pickle("data/raw/games.pkl")
    print("Cargado desde Pickle.")

print(df.shape)
df

Cargado desde Parquet.
(122611, 43)


,app_id,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.00,0,,,,,...,0,0,0 - 0,0,0,0,0,0,0,[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,...,252,3,0 - 20000,8,0,8,0,65,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,...,21,3,0 - 20000,0,0,0,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,...,0,0,0 - 20000,0,0,0,0,0,1,[]
4,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,...,0,0,0 - 20000,0,0,0,0,0,0,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122606,4152910,完美传奇,"Jan 4, 2026",0,0.00,0,《完美传奇》游戏介绍 🔥【完美传奇——专属大陆的奇幻史诗！】🔥 🌍耗时两年匠心打造，诚意之作...,《完美传奇》游戏介绍 🔥【完美传奇——专属大陆的奇幻史诗！】🔥 🌍耗时两年匠心打造，诚意之作...,欢迎来到《完美传奇》！ 🌍专属大陆×自由探索，六道轮回剧情×百变BUFF搭配，一刀爆神装、机...,,...,0,0,0 - 0,0,0,0,0,0,0,[]
122607,4042800,Poker Fate - ACG Texas Hold'em,"Jan 3, 2026",0,0.00,0,Poker Fate – Choose your poker partner and ent...,Poker Fate – Choose your poker partner and ent...,Poker Fate is an anime-style Texas Hold'em gam...,,...,0,0,0 - 0,0,0,0,0,0,0,[]
122608,3522550,Adira Nusantara,"Jan 3, 2026",0,7.99,0,(Gif character) Adira Nusantara is a side-scro...,(Gif character) Adira Nusantara is a side-scro...,"Master authentic Silat combat as Adira, a fier...",,...,0,0,0 - 0,0,0,0,0,0,0,[]
122609,3680350,A Lenda de Niterói,"Jan 4, 2026",0,2.09,0,"Step into the role of Arariboia, a legendary I...","Step into the role of Arariboia, a legendary I...",Embark on Arariboia’s journey during the 16th-...,,...,0,0,0 - 0,0,0,0,0,0,0,[]


In [8]:
COLUMNS = [
    "tags",
    "genres",
    "categories"
]

In [49]:
import numpy as np

def is_empty(value):
    if value is None:
        return True

    if isinstance(value, float) and np.isnan(value):
        return True

    if isinstance(value, (list, tuple)):
        return len(value) == 0

    if isinstance(value, np.ndarray):
        return value.size == 0

    if isinstance(value, str):
        return value.strip() == "" or value.strip() == "[]"

    return False

In [ ]:
# Mask for completely empty rows
mask_all_empty = df.apply(
    lambda row: all(is_empty(row[col]) for col in COLUMNS),
    axis=1
)

# Filter those games
games_without_metadata = df[mask_all_empty]

print("📊 Juegos sin metadata en genres, tags y categories:")
print("Total:", len(games_without_metadata))

# Show only names (first 50)
print("\n🎮 Primeros juegos sin metadata:")
games_without_metadata[["app_id", "name", "tags", "genres", "categories"]].head(50)

📊 Juegos sin metadata en genres, tags y categories:
Total: 7984

🎮 Primeros juegos sin metadata:


,app_id,name,tags,genres,categories
0,2539430,Black Dragon Mage Playtest,[],[],[]
10,1946890,Codename: Warlock Playtest,[],[],[]
24,2349750,CyberVault Playtest,[],[],[]
25,2628280,A Night In Omar's Burger Playtest,[],[],[]
51,2689730,Dark and Deep Playtest,[],[],[]
97,2782210,Bill's Misfortune Playtest,[],[],[]
102,3664570,QUADRA CLASH Playtest,[],[],[]
116,3581240,Descendance Playtest,[],[],[]
118,1549730,Red Solstice 2: Survivors Playtest,[],[],[]
137,1639330,Pain Party Playtest,[],[],[]


# ESTO METER EN DATA CLEAN

In [66]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import ast

COLUMNS = ["genres", "tags", "categories"]

def clean_value(value):
    if value is None:
        return ""

    # NaN
    if isinstance(value, float) and np.isnan(value):
        return ""

    # ndarray
    if isinstance(value, np.ndarray):
        return " ".join(str(i).strip() for i in value if str(i).strip())

    # list
    if isinstance(value, list):
        return " ".join(str(i).strip() for i in value if str(i).strip())

    # dict
    if isinstance(value, dict):
        return " ".join(str(k).strip() for k in value.keys() if str(k).strip())

    # string
    if isinstance(value, str):
        text = value.strip()
        if text == "" or text == "[]":
            return ""

        # 🔥 If it looks like a list or dict, try parsing
        if text.startswith("[") or text.startswith("{"):
            try:
                parsed_value = ast.literal_eval(text)
                if isinstance(parsed_value, dict):
                    return " ".join(str(k).strip() for k in parsed_value.keys() if str(k).strip())
                if isinstance(parsed_value, list):
                    return " ".join(str(i).strip() for i in parsed_value if str(i).strip())
            except:
                pass  # if parsing fails, continue

        return text

    return ""


def remove_duplicates(text):
    words = text.split()
    return " ".join(dict.fromkeys(words))  # keeps order and removes duplicates


def prepare_text(df):
    df = df.copy()
    clean_columns = []

    for col in COLUMNS:
        if col in df.columns:
            series = (
                df[col]
                .map(clean_value)   # faster than apply
                .fillna("")
                .astype(str)
                .str.strip()
            )
            clean_columns.append(series)

    if not clean_columns:
        df["combined_features"] = ""
        return df

    # vectorized concatenation
    combined = clean_columns[0]
    for series in clean_columns[1:]:
        combined = combined.str.cat(series, sep=" ")

    # clean extra spaces
    df["combined_features"] = (
        combined
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .map(remove_duplicates)
    )

    return df


df = prepare_text(df)

print("📊 Filas con combined_features vacío:", (df["combined_features"] == "").sum())

# Opcional: ver cuáles son
print(df[df["combined_features"] == ""][["app_id", "name"]].head(20))

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(df["combined_features"])

📊 Filas con combined_features vacío: 7956
        app_id                                           name
19700  1702710                           Monsty Corp Playtest
19699  3543990                             BIO Fault Playtest
10     1946890                     Codename: Warlock Playtest
19724  3834550                       Legacy of Valor Playtest
19723  3545430                                  妖精之地 Playtest
19719  1625310                        Crab Champions Playtest
19714  4072090  Immortal: And The Death That Follows Playtest
19729  4137000                       Field of Heroes Playtest
19749  2878850                           Trope Tales Playtest
19762  1693080      Kings Gauntlet: Chess Revolution Playtest
19692  3782810                                Cairns Playtest
19783  1987580                            Case #1472 Playtest
19779  3377580                              Cubetory Playtest
19801  1824270                    Creator's Asteroid Playtest
19789  3359150              

# PRUEBAS COMBINAR DATOS

In [69]:
# 🔥 Eliminar juegos duplicados (mismo nombre)
df = df.sort_values("peak_ccu", ascending=False)
df = df.drop_duplicates(subset="name")

In [73]:
import re

def audit_combined_features(df):

    print("🔎 INICIANDO AUDITORÍA\n")

    total_rows = len(df)

    # 1️⃣ Tipo incorrecto
    not_string = df[~df["combined_features"].apply(lambda x: isinstance(x, str))]
    print("❌ Filas que NO son string:", len(not_string))

    # 2️⃣ Contienen sintaxis de dict/lista sin limpiar
    pattern_structure = r"[\{\}\[\]:]"
    weird_structure = df[df["combined_features"].str.contains(pattern_structure, regex=True, na=False)]
    print("❌ Filas con `{ }`, `[ ]` o `:` :", len(weird_structure))

    # 3️⃣ Dobles espacios
    double_spaces = df[df["combined_features"].str.contains(r"\s{2,}", regex=True, na=False)]
    print("⚠️ Filas con dobles espacios:", len(double_spaces))

    # 4️⃣ Texto literal problemático
    weird_text = df[df["combined_features"].str.contains(r"\b(None|nan)\b", regex=True, na=False)]
    print("❌ Filas con 'None' o 'nan' literal:", len(weird_text))

    # 5️⃣ Vacías
    empty_rows = df[df["combined_features"].str.strip() == ""]
    print("ℹ️ Filas vacías:", len(empty_rows))

    print("\n📊 RESUMEN:")
    print("Total filas:", total_rows)
    print("Porcentaje vacías:", round(len(empty_rows)/total_rows*100, 2), "%")

    print("\n🔎 Ejemplos problemáticos (si existen):")

    if len(weird_structure) > 0:
        print("\nEjemplo estructura rara:")
        print(weird_structure["combined_features"].head(3))

    if len(double_spaces) > 0:
        print("\nEjemplo dobles espacios:")
        print(double_spaces["combined_features"].head(3))

    if len(weird_text) > 0:
        print("\nEjemplo texto raro:")
        print(weird_text["combined_features"].head(3))

    print("\n✅ Auditoría finalizada.")

In [75]:
auditar_combined_features(df)

🔎 INICIANDO AUDITORÍA

❌ Filas que NO son string: 0
❌ Filas con `{ }`, `[ ]` o `:` : 0
⚠️ Filas con dobles espacios: 0
❌ Filas con 'None' o 'nan' literal: 0
ℹ️ Filas vacías: 7956

📊 RESUMEN:
Total filas: 121455
Porcentaje vacías: 6.55 %

🔎 Ejemplos problemáticos (si existen):

✅ Auditoría finalizada.


C:\Users\guarm\AppData\Local\Temp\ipykernel_42732\1306733024.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["combined_features"].str.contains(r"\b(None|nan)\b", regex=True, na=False)


In [76]:
import numpy as np
import pandas as pd

def build_user_vector(df, user_games, X):

    if X is None:
        return None

    # Asegurarse de que app_id sea numérico
    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce")

    user_appids = user_games["appid"].astype(int).tolist()
    indices = df[df["app_id"].isin(user_appids)].index

    if len(indices) == 0:
        return None

    # Promedio de los vectores de los juegos del usuario
    user_vector = X[indices].mean(axis=0)

    # 🔥 Convertimos a array normal
    user_vector = np.asarray(user_vector)

    return user_vector

al recomendar un juego asegurarme que no recomiende uno que el usuario ya tenga

In [77]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def recommend(df, X, user_vector, user_games, top_n=10):

    if X is None or user_vector is None:
        print("❌ X o user_vector son None")
        return pd.DataFrame()

    # Asegurar que app_id sea numérico
    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce")

    # Calcular similitud
    similarities = cosine_similarity(user_vector, X)
    scores = similarities.flatten()

    df = df.copy()
    df["similarity"] = scores

    # 🔹 Obtener juegos del usuario
    user_appids = set(user_games["appid"].astype(int))

    # 🔹 No recomendar juegos que el usuario ya posee
    recommendations = df[~df["app_id"].isin(user_appids)]

    # Ordenar por similitud
    recommendations = recommendations.sort_values("similarity", ascending=False)

    return recommendations.head(top_n)

In [78]:
print(df["combined_features"].head())

45509     Action Free To Play FPS Shooter Multiplayer Co...
4193      Action Strategy Free To Play to MOBA Multiplay...
8878      Action Adventure Massively Multiplayer Free To...
104347    Casual Indie Massively Multiplayer Simulation ...
17581     Action RPG Souls-like Online Co-Op Rogue-like ...
Name: combined_features, dtype: str


In [79]:
df["app_id"]

45509         730
4193          570
8878       578080
104347    3419430
17581     2622380
           ...   
122588    4195630
122587    3658370
122586    4259340
122585    4204040
122584    4260870
Name: app_id, Length: 121455, dtype: str

In [83]:
user_games = get_user_games(76561198215235853)
user_games

NameError: name 'get_user_games' is not defined

In [24]:
print("Tipo appID en dataframe:", df["app_id"].dtype)

print("Tipo primer appid usuario:", type(juegos_usuario.iloc[0]["appid"]))

Tipo appID en dataframe: str
Tipo primer appid usuario: <class 'numpy.int64'>


In [25]:
construir_perfil_usuario(df, juegos_usuario)

2026-03-18 09:44:31,139 - INFO - Juegos con al menos 120 minutos jugados: 29
2026-03-18 09:44:31,139 - INFO - AppIDs válidos: 29
2026-03-18 09:44:31,144 - INFO - Juegos encontrados en dataset: 28


,app_id,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,combined_features
1423,2246340,Monster Hunter Wilds,"Feb 27, 2025",0,38.49,190,Deluxe Edition and Premium Deluxe Edition ■Mon...,The unbridled force of nature runs wild and re...,The unbridled force of nature runs wild and re...,,...,91142,20000000 - 50000000,4590,515,3250,281,45,12494,"{'Hunting': 2273, 'Action': 1946, 'Multiplayer...",Action Adventure RPG Hunting Multiplayer Onlin...
4117,1097150,Fall Guys,"Aug 3, 2020",0,0.00,0,You’re invited to dive and dodge your way to v...,You’re invited to dive and dodge your way to v...,"Fall Guys is a free, cross-platform massively ...",,...,88589,20000000 - 50000000,1512,81,553,55,0,664,"{'Multiplayer': 1604, 'Party Game': 1539, 'Bat...",Action Casual Indie Massively Multiplayer Spor...
8773,513710,SCUM,"Jun 17, 2025",0,17.99,13,1.0 Launch Road to 1.0 and beyond! Check out w...,Welcome prisoners to SCUM Island! Where partic...,"Traverse punishing environments, looting, craf...","“Scum has a rare crafting system, a wonder of ...",...,28411,5000000 - 10000000,5771,1375,888,1534,60,8535,"{'Survival': 1480, 'Open World Survival Craft'...",Action Adventure Indie Massively Multiplayer S...
8878,578080,PUBG: BATTLEGROUNDS,"Dec 21, 2017",13,0.00,0,LAND Drop into an ever-growing and changin...,LAND Drop into an ever-growing and changin...,"PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",,...,1037487,100000000 - 200000000,23787,730,6082,302,0,314682,"{'Survival': 14893, 'Shooter': 12788, 'Battle ...",Action Adventure Massively Multiplayer Free To...
14153,346110,ARK: Survival Evolved,"Aug 27, 2017",0,9.89,18,ARK: Survival Ascended! Respawn into a new din...,"As a man or woman stranded naked, freezing and...","Stranded on the shores of a mysterious island,...",,...,117993,20000000 - 50000000,11685,940,921,627,34,22170,"{'Open World Survival Craft': 12961, 'Survival...",Action Adventure Indie Massively Multiplayer R...
17066,252950,Rocket League®,"Jul 6, 2015",0,0.00,0,Rocket League is a high-powered hybrid of arca...,Rocket League is a high-powered hybrid of arca...,Rocket League is a high-powered hybrid of arca...,,...,73775,10000000 - 20000000,19487,491,2807,249,0,21106,"{'Multiplayer': 6610, 'Football (Soccer)': 524...",Action Indie Racing Sports Multiplayer Footbal...
17645,389730,TEKKEN 7,"Jun 1, 2017",0,9.99,24,GET READY Definitive Edition Includes: - TEKKE...,Discover the epic conclusion of the Mishima cl...,Discover the epic conclusion of the long-time ...,,...,15580,2000000 - 5000000,3473,81,551,41,75,599,"{'Fighting': 1279, 'Action': 675, 'Multiplayer...",Action Sports Fighting Multiplayer Competitive...
21033,1599340,Lost Ark,"Feb 11, 2022",17,0.00,7,Embark on an odyssey for the Lost Ark in a vas...,Embark on an odyssey for the Lost Ark in a vas...,Embark on an odyssey for the Lost Ark in a vas...,,...,58540,50000000 - 100000000,5246,1846,746,335,0,17355,"{'MMORPG': 438, 'Free to Play': 380, 'Action R...",Action Adventure Massively Multiplayer RPG Fre...
24982,502500,ACE COMBAT™ 7: SKIES UNKNOWN,"Jan 31, 2019",0,4.79,21,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,Purchase ACE COMBAT™ 7: SKIES UNKNOWN and get ...,Become an ace pilot and soar through photoreal...,,...,5990,2000000 - 5000000,1336,200,429,156,92,340,"{'Flight': 845, 'Jet': 643, 'Military': 610, '...",Action Simulation Flight Jet Military War Shoo...
35608,275850,No Man's Sky,"Aug 12, 2016",0,23.99,1,Inspired by the adventure and imagination that...,Inspired by the adventure and imagination that...,No Man's Sky is a game about exploration and s...,“utterly breathtaking” 9/10 – GameSpot “Soulfu...,...,59526,5000000 - 10000000,3401,810,1239,144,0,12291,"{'Open World': 5393, 'Open World Survival Craf...",Action Adventure Open World Survival Craft Spa...


In [47]:
# =====================================================
# EJEMPLO DE USO
# =====================================================

steam_url = "https://steamcommunity.com/profiles/76561198367022349/"

if __name__ == "__main__":

    import pandas as pd
    
    # 1️⃣ Cargar dataset
    print("🔄 Cargando dataset...")
    
    # 2️⃣ Preparar texto y matriz TF-IDF
    print("🧠 Preparando features...")
    df = preparar_texto(df)

    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform(df["combined_features"])

    # 3️⃣ Pedir URL de Steam
    url_usuario = input("\nIntroduce tu perfil de Steam: ")

    steamid = extraer_steamid(url_usuario)

    if not steamid:
        print("❌ No se pudo obtener el SteamID.")
        exit()

    print("✅ SteamID detectado:", steamid)

    # 4️⃣ Obtener juegos del usuario
    juegos_usuario = obtener_juegos_usuario(steamid)

    if juegos_usuario.empty:
        print("⚠️ Perfil privado o sin juegos.")
        exit()

    print(f"🎮 Juegos encontrados: {len(juegos_usuario)}")

    # 5️⃣ Construir vector del usuario
    user_vector = construir_vector_usuario(df, juegos_usuario, X)

    if user_vector is None:
        print("⚠️ No se pudieron mapear los juegos al dataset.")
        exit()
    else: 
        print("Conseguido")

    # 6️⃣ Generar recomendaciones
    recomendaciones = recomendar(df, X, user_vector, juegos_usuario, top_n=10)

    print("\n🎯 RECOMENDACIONES PERSONALIZADAS:\n")

    for i, row in recomendaciones.iterrows():
        print(f"🎮 {row['name']}")
        print(f"   Similaridad: {row['similarity']:.4f}")
        print(f"   Géneros: {row['genres']}")
        print(f"   Tags: {row['tags']}")
        print(f"   Categorias: {row['categories']}")


        print("-" * 50)

🔄 Cargando dataset...
🧠 Preparando features...


2026-03-18 10:05:35,420 - INFO - SteamID numérico detectado
2026-03-18 10:05:35,421 - INFO - Consultando juegos del usuario 76561198367022349
2026-03-18 10:05:35,422 - INFO - Enviando petición a la API de Steam


✅ SteamID detectado: 76561198367022349


2026-03-18 10:05:35,895 - INFO - Código de respuesta: 200
2026-03-18 10:05:35,895 - INFO - Se encontraron 76 juegos


🎮 Juegos encontrados: 76
Conseguido

🎯 RECOMENDACIONES PERSONALIZADAS:

🎮 Demonbyte
   Similaridad: 0.7540
   Géneros: ['Action' 'Adventure' 'Casual' 'Indie']
   Tags: []
   Categorias: ['Single-player' 'Steam Achievements' 'Full controller support'
 'Family Sharing']
--------------------------------------------------
🎮 Arcane Tower Survivors
   Similaridad: 0.7540
   Géneros: ['Action' 'Adventure' 'Casual' 'Indie']
   Tags: []
   Categorias: ['Single-player' 'Steam Achievements' 'Full controller support'
 'Family Sharing']
--------------------------------------------------
🎮 Jajazinho e as Delicias de Cristais
   Similaridad: 0.7540
   Géneros: ['Action' 'Adventure' 'Casual' 'Indie']
   Tags: []
   Categorias: ['Single-player' 'Steam Achievements' 'Full controller support'
 'Family Sharing']
--------------------------------------------------
🎮 rock climbing
   Similaridad: 0.7540
   Géneros: ['Action' 'Adventure' 'Casual' 'Indie']
   Tags: []
   Categorias: ['Single-player' 'Steam Ach

In [44]:
url_usuario = "https://steamcommunity.com/profiles/76561198367022349"

print("🔄 Preparando texto...")
df = preparar_texto(df)
print("✅ combined_features creado")

print("🧠 Creando TF-IDF...")
vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(df["combined_features"])
print(f"✅ TF-IDF listo. Shape: {X.shape}")

print("\n🔗 Obteniendo SteamID...")
url_usuario = input("Introduce tu URL de Steam: ")
steamid = extraer_steamid(url_usuario)

if not steamid:
    print("❌ No se pudo obtener SteamID")
    exit()

print(f"✅ SteamID: {steamid}")

print("🎮 Obteniendo juegos del usuario...")
juegos_usuario = obtener_juegos_usuario(steamid)

if not juegos_usuario:
    print("⚠️ No se encontraron juegos o perfil privado")
    exit()

print(f"✅ Juegos encontrados: {len(juegos_usuario)}")

print("🧠 Construyendo vector del usuario...")
user_vector = construir_vector_usuario(df, juegos_usuario, X)

if user_vector is None:
    print("❌ No se pudo construir vector del usuario")
    exit()

print("✅ Vector del usuario creado")

print("🎯 Generando recomendaciones...")
recomendaciones = recomendar(df, X, user_vector, juegos_usuario)

if recomendaciones is None or recomendaciones.empty:
    print("⚠️ No se encontraron recomendaciones")
else:
    print("\n🎉 RECOMENDACIONES:")
    print(recomendaciones[["name", "similarity"]].head(10))

🔄 Preparando texto...
✅ combined_features creado
🧠 Creando TF-IDF...
✅ TF-IDF listo. Shape: (122611, 544)

🔗 Obteniendo SteamID...


2026-03-15 22:07:42,331 - INFO - SteamID numérico detectado
2026-03-15 22:07:42,333 - INFO - Consultando juegos del usuario 76561198367022349
2026-03-15 22:07:42,333 - INFO - Enviando petición a la API de Steam


✅ SteamID: 76561198367022349
🎮 Obteniendo juegos del usuario...


2026-03-15 22:07:42,630 - INFO - Código de respuesta: 200
2026-03-15 22:07:42,631 - INFO - Se encontraron 76 juegos


ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
import numpy as np

def construir_vector_usuario(df, juegos_usuario, feature_matrix):
    
    appids_usuario = [j["appid"] for j in juegos_usuario]
    
    indices = df[df["appID"].isin(appids_usuario)].index
    
    if len(indices) == 0:
        return None
    
    # Vector promedio
    user_vector = np.asarray(feature_matrix[indices].mean(axis=0))
    
    return user_vector

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df.genres)